# GAN Example 3 (PyTorch) — A 2D Toy GAN

PyTorch version of `toy_2d_gan.ipynb`. The 'real data' is a ring of **8 Gaussian blobs**;
we plot real vs fake points so you can watch the generator learn the distribution — and
spot **mode collapse** (covering only some blobs) directly.

Tiny MLPs ⇒ trains in seconds on CPU.

> Needs `torch`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

In [ ]:
NOISE_DIM = 2
centers = np.array([[np.cos(t), np.sin(t)] for t in np.linspace(0, 2*np.pi, 8, endpoint=False)],
                   dtype='float32') * 2.0

def sample_real(n):
    idx = np.random.randint(0, 8, size=n)
    return torch.tensor((centers[idx] + 0.1 * np.random.randn(n, 2)).astype('float32'))

real_np = sample_real(2000).numpy()
plt.figure(figsize=(4, 4))
plt.scatter(real_np[:, 0], real_np[:, 1], s=4)
plt.title('Target distribution (8 Gaussians)'); plt.axis('equal'); plt.show()

## Models

In [ ]:
def mlp(out):
    return nn.Sequential(nn.Linear(2, 128), nn.ReLU(),
                         nn.Linear(128, 128), nn.ReLU(),
                         nn.Linear(128, out))

G = mlp(2).to(device)     # noise -> point
D = mlp(1).to(device)     # point -> logit
criterion = nn.BCEWithLogitsLoss()
g_opt = torch.optim.Adam(G.parameters(), lr=1e-3, betas=(0.5, 0.999))
d_opt = torch.optim.Adam(D.parameters(), lr=1e-3, betas=(0.5, 0.999))

def plot_compare(step):
    G.eval()
    with torch.no_grad():
        fake = G(torch.randn(2000, NOISE_DIM, device=device)).cpu().numpy()
    G.train()
    plt.figure(figsize=(4, 4))
    plt.scatter(real_np[:, 0], real_np[:, 1], s=4, alpha=0.3, label='real')
    plt.scatter(fake[:, 0], fake[:, 1], s=4, alpha=0.3, label='fake')
    plt.legend(); plt.axis('equal'); plt.title(f'Step {step}'); plt.show()

## Train

In [ ]:
STEPS = 5000
for step in range(1, STEPS + 1):
    real = sample_real(256).to(device)
    bsz = real.size(0)
    ones, zeros = torch.ones(bsz, 1, device=device), torch.zeros(bsz, 1, device=device)

    # Discriminator
    z = torch.randn(bsz, NOISE_DIM, device=device)
    fake = G(z)
    d_loss = criterion(D(real), ones) + criterion(D(fake.detach()), zeros)
    d_opt.zero_grad(); d_loss.backward(); d_opt.step()

    # Generator
    g_loss = criterion(D(fake), ones)
    g_opt.zero_grad(); g_loss.backward(); g_opt.step()

    if step % 1000 == 0 or step == 1:
        print(f'Step {step:5d}/{STEPS}  gen_loss={g_loss.item():.3f}  disc_loss={d_loss.item():.3f}')
        plot_compare(step)

### Things to try

- Shrink the generator to induce mode collapse.
- Change the target to a spiral or two moons.
- Add multiple discriminator steps per generator step.
- Use this sandbox to test WGAN / gradient penalty against collapse.